# DVF 2022 — Lignes rendues identiques par l'expurgation

Le fichier DVF represente chaque bien concerne par une mutation
sur une ligne distincte (notice descriptive, DGFiP, 2022, p. 3).
Si une vente porte sur deux garages, le fichier contient deux lignes.
Ces deux garages sont des biens physiquement differents.

Cependant, la colonne Identifiant local — qui distinguait chaque
bien de maniere unique — a ete supprimee (decret 2018-1350).
Sans cet identifiant, deux garages identiques dans le meme immeuble
deviennent des **lignes indistinguables** : meme date, meme prix,
meme commune, meme surface, meme type.

Ce notebook identifie et illustre ces lignes rendues identiques.

## Cellule 1 — Connexion et preparation

In [1]:
import duckdb
from pathlib import Path

FICHIER = Path(r"./data/dvf-2022.parquet")
con = duckdb.connect()
pq = str(FICHIER)

nb_lignes = con.execute(f"SELECT count(*) FROM '{pq}'").fetchone()[0]
colonnes = con.execute(f"DESCRIBE SELECT * FROM '{pq}'").fetchall()
col_names = [c[0] for c in colonnes]
cols_vides = []
for c in col_names:
    n = con.execute(f'SELECT count(*) FROM \'{pq}\' WHERE "{c}" IS NULL').fetchone()[0]
    if n == nb_lignes:
        cols_vides.append(c)
cols_exp = [c for c in col_names if c not in cols_vides]
cols_sql = ', '.join([f'"' + c + '"' for c in cols_exp])

print(f"Fichier : {FICHIER.name}")
print(f"Lignes : {nb_lignes:,}".replace(',', ' '))
print(f"Colonnes exploitables : {len(cols_exp)}")

Fichier : dvf-2022.parquet
Lignes : 4 617 590
Colonnes exploitables : 35


## Cellule 2 — Combien de lignes sont indistinguables ?

Regrouper les lignes par les 35 colonnes exploitables.
Si un groupe contient plus d'une ligne, ces lignes sont
indistinguables entre elles.

In [2]:
stats = con.execute(f"""
    SELECT
        count(*) AS nb_groupes,
        sum(n) AS lignes_impliquees,
        sum(n - 1) AS lignes_sans_identifiant_unique
    FROM (
        SELECT count(*) AS n
        FROM '{pq}'
        GROUP BY {cols_sql}
        HAVING count(*) > 1
    )
""").fetchone()

print("Lignes rendues identiques par l'expurgation")
print("=" * 55)
print(f"  Groupes de lignes identiques : {stats[0]:>10,}".replace(",", " "))
print(f"  Lignes dans ces groupes      : {stats[1]:>10,}".replace(",", " "))
print(f"  Dont sans identifiant unique : {stats[2]:>10,}".replace(",", " "))
print(f"  Taux                         : {100*stats[2]/nb_lignes:.2f} %")
print()
print("Ces lignes representent des biens physiquement differents")
print("(deux garages distincts, par exemple) qui sont devenus")
print("indistinguables apres la suppression de l'Identifiant local.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Lignes rendues identiques par l'expurgation
  Groupes de lignes identiques :    191 468
  Lignes dans ces groupes      :    536 750
  Dont sans identifiant unique :    345 282
  Taux                         : 7.48 %

Ces lignes representent des biens physiquement differents
(deux garages distincts, par exemple) qui sont devenus
indistinguables apres la suppression de l'Identifiant local.


## Cellule 3 — Exemple 1 : un groupe de 2 lignes identiques

Trouver un groupe ou 2 lignes ont exactement les memes valeurs
sur les 35 colonnes. Ce sont probablement deux dependances
identiques (garages, caves) dans le meme immeuble.

In [3]:
cols_aff = ["Date mutation", "Nature mutation", "Valeur fonciere",
            "Code departement", "Commune", "Type local",
            "Surface reelle bati", "Nombre pieces principales",
            "Surface terrain", "Nature culture", "Nombre de lots"]

groupe2 = con.execute(f"""
    SELECT {cols_sql}
    FROM '{pq}'
    GROUP BY {cols_sql}
    HAVING count(*) = 2
    ORDER BY "Date mutation", "Commune"
    LIMIT 1
""").fetchdf()

print("Groupe de 2 lignes identiques :")
print()
for c in cols_aff:
    if c in groupe2.columns:
        val = groupe2[c].iloc[0]
        if val is None:
            val = "(vide)"
        print(f"  {c:<30} {val}")
print()
print("Ces 2 lignes representent 2 biens distincts (par exemple,")
print("2 garages dans le meme immeuble) rendus indistinguables")
print("par la suppression de l'Identifiant local.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Groupe de 2 lignes identiques :

  Date mutation                  2022-01-03 00:00:00
  Nature mutation                Vente
  Valeur fonciere                96700
  Code departement               34
  Commune                        AGDE
  Type local                     Dépendance
  Surface reelle bati            0
  Nombre pieces principales      0
  Surface terrain                <NA>
  Nature culture                 (vide)
  Nombre de lots                 1

Ces 2 lignes representent 2 biens distincts (par exemple,
2 garages dans le meme immeuble) rendus indistinguables
par la suppression de l'Identifiant local.


## Cellule 4 — Exemple 2 : un groupe de 5 lignes identiques

In [4]:
groupe5 = con.execute(f"""
    SELECT {cols_sql}
    FROM '{pq}'
    GROUP BY {cols_sql}
    HAVING count(*) = 5
    ORDER BY "Date mutation", "Commune"
    LIMIT 1
""").fetchdf()

print("Groupe de 5 lignes identiques :")
print()
for c in cols_aff:
    if c in groupe5.columns:
        val = groupe5[c].iloc[0]
        if val is None:
            val = "(vide)"
        print(f"  {c:<30} {val}")
print()
print("Ces 5 lignes representent 5 biens distincts (par exemple,")
print("5 places de parking dans le meme immeuble) rendus")
print("indistinguables par la suppression de l'Identifiant local.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Groupe de 5 lignes identiques :

  Date mutation                  2022-01-03 00:00:00
  Nature mutation                Vente
  Valeur fonciere                75000
  Code departement               35
  Commune                        GUIPRY-MESSAC
  Type local                     Dépendance
  Surface reelle bati            0
  Nombre pieces principales      0
  Surface terrain                1000
  Nature culture                 S
  Nombre de lots                 0

Ces 5 lignes representent 5 biens distincts (par exemple,
5 places de parking dans le meme immeuble) rendus
indistinguables par la suppression de l'Identifiant local.


## Cellule 5 — Le plus grand groupe

In [5]:
max_n = con.execute(f"""
    SELECT max(n) FROM (
        SELECT count(*) AS n FROM '{pq}' GROUP BY {cols_sql} HAVING count(*) > 1
    )
""").fetchone()[0]

plus_grand = con.execute(f"""
    SELECT {cols_sql}
    FROM '{pq}'
    GROUP BY {cols_sql}
    HAVING count(*) = {max_n}
    LIMIT 1
""").fetchdf()

print(f"Le plus grand groupe : {max_n} lignes identiques")
print()
for c in cols_aff:
    if c in plus_grand.columns:
        val = plus_grand[c].iloc[0]
        if val is None:
            val = "(vide)"
        print(f"  {c:<30} {val}")
print()
print(f"Ces {max_n} lignes representent {max_n} biens distincts")
print(f"(probablement {max_n} places de parking) rendus indistinguables.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Le plus grand groupe : 976 lignes identiques

  Date mutation                  2022-12-27 00:00:00
  Nature mutation                Vente
  Valeur fonciere                314985152
  Code departement               56
  Commune                        VANNES
  Type local                     Dépendance
  Surface reelle bati            0
  Nombre pieces principales      0
  Surface terrain                52195
  Nature culture                 S
  Nombre de lots                 0

Ces 976 lignes representent 976 biens distincts
(probablement 976 places de parking) rendus indistinguables.


## Cellule 6 — Distribution par taille de groupe

In [6]:
distrib = con.execute(f"""
    SELECT n AS taille, count(*) AS nb_groupes, count(*) * n AS nb_lignes
    FROM (
        SELECT count(*) AS n FROM '{pq}' GROUP BY {cols_sql} HAVING count(*) > 1
    )
    GROUP BY n ORDER BY n
""").fetchall()

print(f"{'Taille du groupe':<20} {'Nb groupes':>12} {'Nb lignes':>12}")
print("-" * 50)
for taille, nb_gr, nb_lig in distrib[:10]:
    print(f"  {taille:<20} {nb_gr:>10,} {nb_lig:>10,}".replace(",", " "))
if len(distrib) > 10:
    print(f"  ... ({len(distrib) - 10} autres tailles)")
    print(f"  {distrib[-1][0]:<20} {distrib[-1][1]:>10,} {distrib[-1][2]:>10,}".replace(",", " "))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Taille du groupe       Nb groupes    Nb lignes
--------------------------------------------------
  2                       143 932    287 864
  3                        27 791     83 373
  4                         9 304     37 216
  5                         3 071     15 355
  6                         1 984     11 904
  7                           876      6 132
  8                           837      6 696
  9                           560      5 040
  10                          458      4 580
  11                          269      2 959
  ... (158 autres tailles)
  976                           1        976


## Cellule 7 — Resume

In [7]:
g2_commune = groupe2["Commune"].iloc[0] if "Commune" in groupe2.columns else "?"
g5_commune = groupe5["Commune"].iloc[0] if "Commune" in groupe5.columns else "?"
bg_commune = plus_grand["Commune"].iloc[0] if "Commune" in plus_grand.columns else "?"

print("Resume")
print("=" * 60)
print(f"{'Exemple':<35} {'Lignes':>10}")
print("-" * 60)
print(f"{'Groupe de 2 (' + str(g2_commune) + ')':<35} {'2':>10}")
print(f"{'Groupe de 5 (' + str(g5_commune) + ')':<35} {'5':>10}")
print(f"{'Plus grand (' + str(bg_commune) + ')':<35} {str(max_n):>10}")
print("-" * 60)
print()
print(f"Total dans le fichier :")
print(f"  {stats[0]:,} groupes de lignes identiques".replace(",", " "))
print(f"  {stats[1]:,} lignes impliquees".replace(",", " "))
print(f"  {stats[2]:,} lignes sans identifiant unique ({100*stats[2]/nb_lignes:.2f} %)".replace(",", " "))
print()
print("Important : ces lignes ne sont pas des erreurs.")
print("Chaque ligne represente un bien reel (un garage, une cave,")
print("un parking). La notice descriptive de la DGFiP (2022, p. 3)")
print("confirme que le fichier comporte 'autant de lignes qu'il y")
print("a de locaux'. La suppression de l'Identifiant local (decret")
print("2018-1350) a rendu ces biens distincts indistinguables.")

Resume
Exemple                                 Lignes
------------------------------------------------------------
Groupe de 2 (AGDE)                           2
Groupe de 5 (GUIPRY-MESSAC)                  5
Plus grand (VANNES)                        976
------------------------------------------------------------

Total dans le fichier :
  191 468 groupes de lignes identiques
  536 750 lignes impliquees
  345 282 lignes sans identifiant unique (7.48 %)

Important : ces lignes ne sont pas des erreurs.
Chaque ligne represente un bien reel (un garage, une cave,
un parking). La notice descriptive de la DGFiP (2022, p. 3)
confirme que le fichier comporte 'autant de lignes qu'il y
a de locaux'. La suppression de l'Identifiant local (decret
2018-1350) a rendu ces biens distincts indistinguables.


In [8]:
con.close()
print("Connexion fermee.")

Connexion fermee.
